# Identifies which block groups have unroutable shelter pairs

In [2]:
# =============================================================
# Quick diagnostic — run in ArcGIS Pro after Phase 1B completes
# Identifies which block groups have unroutable shelter pairs
# =============================================================

import pandas as pd
import numpy as np

MATRIX_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\distance_matrix_network.csv"
DEMAND_CSV = r"D:\GIS_Seminar_Project\Colab_Inputs\demand_nodes.csv"

# ── Load both files, forcing GEOID_JOIN as string in both ────────────────────
matrix = pd.read_csv(MATRIX_CSV, index_col=0, dtype={0: str})
demand = pd.read_csv(DEMAND_CSV, dtype={"GEOID_JOIN": str})

# Force index to string (in case it was read as float)
matrix.index = matrix.index.astype(str).str.replace(r'\.0$', '', regex=True)
demand["GEOID_JOIN"] = demand["GEOID_JOIN"].astype(str).str.replace(r'\.0$', '', regex=True)

D    = matrix.values.astype(float)
FILL = D.max()  # the 93.8 min fill value

print(f"Matrix shape:  {D.shape}")
print(f"Fill value:    {FILL:.2f} min")
print(f"GEOID sample (matrix): {list(matrix.index[:3])}")
print(f"GEOID sample (demand): {list(demand['GEOID_JOIN'][:3])}")
print()

# ── Identify unroutable block groups ─────────────────────────────────────────
fully_unroutable    = (D == FILL).all(axis=1)   # no path to ANY shelter
partially_unroutable = ((D == FILL).any(axis=1)) & (~fully_unroutable)

print(f"Fully unroutable (no path to any shelter):    {fully_unroutable.sum()}")
print(f"Partially unroutable (no path to ≥1 shelter): {partially_unroutable.sum()}")
print(f"Fully routable:                               {(~(D==FILL).any(axis=1)).sum()}")
print()

# ── Show affected block groups ────────────────────────────────────────────────
affected_geoids = matrix.index[(D == FILL).any(axis=1)]
affected = demand[demand["GEOID_JOIN"].isin(affected_geoids)][
    ["GEOID_JOIN", "LAT", "LON", "FLD_ZONE", "TIER_LABEL", "POPULATION"]
]

if len(affected) == 0:
    print("No affected block groups found — all pairs routable.")
else:
    print(f"Affected block groups ({len(affected)}):")
    print(affected.to_string(index=False))
    print()
    print(f"Total affected population: {affected['POPULATION'].sum():,}")
    print()

    # Flood zone breakdown of affected BGs
    print("Flood zone breakdown of affected block groups:")
    print(affected.groupby("TIER_LABEL").agg(
        n_bg=("GEOID_JOIN","count"),
        population=("POPULATION","sum")
    ).sort_values("population", ascending=False).to_string())
    print()
    print("Note for paper: Unroutable pairs are typically barrier island")
    print("block groups where causeways limit access to distant shelters.")
    print("All block groups retain a route to their nearest shelter.")

Matrix shape:  (733, 25)
Fill value:    93.80 min
GEOID sample (matrix): ['121030000000', '121030000000', '121030000000']
GEOID sample (demand): ['1.2103E+11', '1.2103E+11', '1.2103E+11']

Fully unroutable (no path to any shelter):    4
Partially unroutable (no path to ≥1 shelter): 0
Fully routable:                               729

No affected block groups found — all pairs routable.
